# GDELT Quarterly Sentiment (1990-2025)

GDELT (Global Database of Events, Language, and Tone)에서
미국 경제 관련 이벤트의 분기별 센티먼트(AvgTone) 추출

- BigQuery 공개 데이터셋, Rate Limit 없음
- AvgTone: 기사 감성 점수 (negative ~ positive)
- GoldsteinScale: 이벤트 협력/갈등 정도 (-10 ~ +10)
- 실행 시간: 약 1분

In [ ]:
from google.cloud import bigquery
import pandas as pd
import subprocess

# Colab 기본 프로젝트 ID 감지
project_id = subprocess.check_output(['gcloud', 'config', 'get-value', 'project']).decode().strip()
if not project_id:
    project_id = input('Google Cloud Project ID: ')

print(f'Project: {project_id}')
client = bigquery.Client(project=project_id)
print('Ready.')

In [ ]:
QUERY = """
SELECT
    EXTRACT(YEAR FROM PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING))) AS year,
    EXTRACT(QUARTER FROM PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING))) AS quarter,
    FORMAT_DATE('%Y-%m', PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING))) AS year_month,

    AVG(AvgTone) AS tone_mean,
    STDDEV(AvgTone) AS tone_std,
    APPROX_QUANTILES(AvgTone, 100)[OFFSET(50)] AS tone_median,
    MIN(AvgTone) AS tone_min,
    MAX(AvgTone) AS tone_max,

    AVG(GoldsteinScale) AS goldstein_mean,
    STDDEV(GoldsteinScale) AS goldstein_std,

    COUNT(*) AS event_count,
    SUM(NumArticles) AS total_articles,
    AVG(NumArticles) AS avg_articles_per_event,

FROM `gdelt-bq.full.events`
WHERE
    SQLDATE >= 19900101
    AND SQLDATE <= 20251231
    AND (
        Actor1CountryCode = 'USA'
        OR Actor2CountryCode = 'USA'
        OR ActionGeo_CountryCode = 'US'
    )
    AND AvgTone IS NOT NULL

GROUP BY year, quarter, year_month
ORDER BY year, quarter, year_month
"""

print('Running query (may take ~1 min)...')
df = client.query(QUERY).to_dataframe()
print(f'Done! {len(df)} rows')
df.head(10)

In [ ]:
# Quarterly aggregation
quarterly = df.groupby(['year', 'quarter']).agg(
    tone_mean=('tone_mean', 'mean'),
    tone_std=('tone_std', 'mean'),
    tone_median=('tone_median', 'median'),
    goldstein_mean=('goldstein_mean', 'mean'),
    goldstein_std=('goldstein_std', 'mean'),
    event_count=('event_count', 'sum'),
    total_articles=('total_articles', 'sum'),
).reset_index()

quarterly['quarter_label'] = quarterly['year'].astype(int).astype(str) + 'Q' + quarterly['quarter'].astype(int).astype(str)
print(f'{len(quarterly)} quarters')
quarterly

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

x = range(len(quarterly))
labels = quarterly['quarter_label'].values

axes[0].plot(x, quarterly['tone_mean'], 'b-', alpha=0.7)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_ylabel('AvgTone (mean)')
axes[0].set_title('GDELT US Events Sentiment (1990-2025, quarterly)')

axes[1].plot(x, quarterly['goldstein_mean'], 'r-', alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_ylabel('GoldsteinScale (mean)')

axes[2].bar(x, quarterly['event_count'], color='steelblue', alpha=0.7)
axes[2].set_ylabel('Event count')

tick_pos = [i for i in x if i % 4 == 0]
tick_labels = [labels[i][:4] for i in tick_pos]
axes[2].set_xticks(tick_pos)
axes[2].set_xticklabels(tick_labels, rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Save & Download
quarterly.to_csv('gdelt_quarterly_sentiment.csv', index=False)
df.to_csv('gdelt_monthly_sentiment.csv', index=False)

from google.colab import files
files.download('gdelt_quarterly_sentiment.csv')
files.download('gdelt_monthly_sentiment.csv')
print('Done!')